# 02 — NutriVision: full training & evaluation pipeline (Colab)

Run top to bottom. GPU runtime recommended (Runtime → Change runtime type → T4 GPU).

## 1. Setup: clone repo + install dependencies

In [ ]:
!git clone https://github.com/YOUR_USERNAME/NutriVision.git
%cd /content/NutriVision
!pip install -q transformers datasets timm grad-cam gradio seaborn

In [ ]:
import sys
sys.path.insert(0, "/content/NutriVision")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Mount Drive so checkpoints survive the session

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
CKPT_DIR = Path("/content/drive/MyDrive/NutriVision/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("/content/drive/MyDrive/NutriVision/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Data (first run downloads Food-101, ~5 GB)

In [ ]:
from src.config import Config
from src.utils import set_seed
from src.data import get_dataloaders, FOOD40

set_seed(Config.SEED)
train_loader, val_loader, test_loader = get_dataloaders()
print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

## 4. Train Model A — BaselineCNN (from scratch)

In [ ]:
from src.model_a import BaselineCNN
from src.train import train, plot_curves

model_a = BaselineCNN()
hist_a = train(
    model_a, train_loader, val_loader,
    epochs=Config.NUM_EPOCHS_A, lr=Config.LR_A,
    scheduler_type="cosine", patience=5,
    checkpoint_path=CKPT_DIR / "model_a.pt",
)
plot_curves(hist_a, OUT_DIR / "model_a_curves.png", title="Model A — BaselineCNN")

## 5. Train Model B — Swin-Tiny (phase 1: frozen backbone, phase 2: full fine-tune)

In [ ]:
from src.model_b import get_model_b, freeze_backbone, unfreeze_all

model_b, processor = get_model_b()

freeze_backbone(model_b)
hist_b1 = train(
    model_b, train_loader, val_loader,
    epochs=Config.FREEZE_EPOCHS_B, lr=1e-3,
    scheduler_type="none", patience=5,
    checkpoint_path=CKPT_DIR / "model_b_frozen.pt",
)

unfreeze_all(model_b)
hist_b2 = train(
    model_b, train_loader, val_loader,
    epochs=Config.NUM_EPOCHS_B - Config.FREEZE_EPOCHS_B, lr=Config.LR_B,
    scheduler_type="cosine", patience=5,
    checkpoint_path=CKPT_DIR / "model_b.pt",
)

hist_b = {k: hist_b1[k] + hist_b2[k] for k in hist_b1}
plot_curves(hist_b, OUT_DIR / "model_b_curves.png", title="Model B — Swin-Tiny")

### Reload best checkpoints (run from here if the session restarted)

In [ ]:
from src.model_a import BaselineCNN
from src.model_b import get_model_b

model_a = BaselineCNN()
model_a.load_state_dict(torch.load(CKPT_DIR / "model_a.pt", map_location="cpu")["model_state_dict"])

model_b, processor = get_model_b()
model_b.load_state_dict(torch.load(CKPT_DIR / "model_b.pt", map_location="cpu")["model_state_dict"])

## 6. Evaluate both models on the test set

In [ ]:
from src.evaluate import evaluate, plot_confusion_matrix, show_worst_predictions

results_a = evaluate(model_a, test_loader, FOOD40)
results_b = evaluate(model_b, test_loader, FOOD40)

import pandas as pd
summary = pd.DataFrame({
    "Model A (CNN)": [results_a["accuracy"], results_a["macro_f1"], results_a["weighted_f1"]],
    "Model B (Swin)": [results_b["accuracy"], results_b["macro_f1"], results_b["weighted_f1"]],
}, index=["Accuracy", "Macro F1", "Weighted F1"])
summary.round(4)

In [ ]:
results_b["per_class"].sort_values("f1").head(10)

In [ ]:
plot_confusion_matrix(results_b["confusion_matrix"], FOOD40, OUT_DIR / "model_b_cm.png")
from IPython.display import Image as IPImage
IPImage(str(OUT_DIR / "model_b_cm.png"), width=900)

In [ ]:
show_worst_predictions(model_b, test_loader, k=10, class_names=FOOD40,
                       save_path=OUT_DIR / "model_b_worst.png")

## 7. Grad-CAM — where does each model look?

In [ ]:
from src.gradcam import visualize_gradcam_grid

images, labels = next(iter(test_loader))
visualize_gradcam_grid(model_a, images[:10], OUT_DIR / "gradcam_cnn.png",
                       class_names=FOOD40, labels=labels[:10])
visualize_gradcam_grid(model_b, images[:10], OUT_DIR / "gradcam_swin.png",
                       class_names=FOOD40, labels=labels[:10])

## 8. MC-dropout uncertainty

In [ ]:
from src.uncertainty import predict_with_uncertainty

img, label = test_loader.dataset[0]
r = predict_with_uncertainty(model_b, img, n_samples=20)
print(f"true: {FOOD40[label]}")
print(f"pred: {FOOD40[r['predicted_class']]} "
      f"p={r['predicted_prob']:.2f} ± {r['predicted_std']:.3f}")
print(f"entropy: {r['entropy']:.2f} -> confidence: {r['confidence']}")

## 9. Depth, segmentation & volume demo

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.volume import estimate_depth, estimate_food_mask, estimate_volume
from src.data import load_food40_splits

_, _, test_raw, _ = load_food40_splits()
pil_img = test_raw[0]["image"].convert("RGB")

depth = estimate_depth(pil_img)
mask = estimate_food_mask(pil_img)
vol = estimate_volume(pil_img, food_mask=mask, depth_map=depth,
                      reference_scale_cm_per_pixel=0.05)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(pil_img); axes[0].set_title("input"); axes[0].axis("off")
axes[1].imshow(depth, cmap="inferno"); axes[1].set_title("depth"); axes[1].axis("off")
axes[2].imshow(pil_img); axes[2].imshow(mask, alpha=0.5, cmap="Greens")
axes[2].set_title(f"mask -> {vol['volume_cm3']:.0f} cm³"); axes[2].axis("off")
plt.show()

## 10. Done

Checkpoints and figures live on Drive (`CKPT_DIR`, `OUT_DIR`), so nothing is lost when the runtime dies.